In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQL Lab").getOrCreate()


employees_data = [
    (1, "Rahul", "IT", 70000, "Hyderabad"),
    (2, "Sneha", "HR", 60000, "Bangalore"),
    (3, "Arjun", "IT", 75000, "Chennai"),
    (4, "Priya", "Finance", 80000, "Hyderabad"),
    (5, "Karan", "IT", 50000, "Mumbai"),
    (6, "Amit", "HR", 58000, "Delhi"),
    (7, "Meera", "Finance", 82000, "Bangalore")
]

employees_cols = ["emp_id", "name", "department", "salary", "city"]

employees_df = spark.createDataFrame(employees_data, employees_cols)
employees_df.show()

departments_data = [
    ("IT", "Technology"),
    ("HR", "People Operations"),
    ("Finance", "Accounts and Finance")
]

departments_cols = ["department", "dept_full_name"]

departments_df = spark.createDataFrame(departments_data, departments_cols)
departments_df.show()

sales_data = [
    (101, 1, "Laptop", 1, 75000),
    (102, 2, "Mouse", 3, 500),
    (103, 3, "Keyboard", 2, 1500),
    (104, 1, "Monitor", 1, 12000),
    (105, 4, "Laptop", 1, 75000),
    (106, 3, "Mouse", 2, 500),
    (107, 5, "Keyboard", 1, 1500),
    (108, 1, "Laptop", 1, 75000)
]

sales_cols = ["sale_id", "emp_id", "product", "quantity", "price"]

sales_df = spark.createDataFrame(sales_data, sales_cols)
sales_df.show()


In [ ]:
employees_df.createOrReplaceTempView("employees")
departments_df.createOrReplaceTempView("departments")
sales_df.createOrReplaceTempView("sales")

In [ ]:
spark.sql("select * from employees").show()

In [ ]:
employees_df.createOrReplaceGlobalTempView("g_employees")

In [ ]:
spark.sql("select * from global_temp.g_employees").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
+------+-----+----------+------+---------+



In [ ]:
spark.sql("create database if not exists company_db")

DataFrame[]

In [ ]:
spark.sql("show databases").show()

+----------+
| namespace|
+----------+
|company_db|
|   default|
+----------+



In [ ]:
spark.sql("use company_db")

DataFrame[]

In [ ]:
spark.sql("select current_database()").show()

+----------------+
|current_schema()|
+----------------+
|      company_db|
+----------------+



In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS employee_master (
    emp_id INT,
    name STRING,
    department STRING,
    salary INT,
    city STRING
)
USING PARQUET
""")

DataFrame[]

In [ ]:
spark.sql("""
INSERT INTO employee_master VALUES
(1, 'Rahul', 'IT', 70000, 'Hyderabad'),
(2, 'Sneha', 'HR', 60000, 'Bangalore'),
(3, 'Arjun', 'IT', 75000, 'Chennai'),
(4, 'Priya', 'Finance', 80000, 'Hyderabad')
""")

DataFrame[]

In [ ]:
spark.sql("select * from employee_master").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
+------+-----+----------+------+---------+



In [ ]:
employees_df.write.mode("overwrite").saveAsTable("employee_details")

In [ ]:
spark.sql("select * from employee_details").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
+------+-----+----------+------+---------+



In [ ]:
spark.sql("show tables in company_db").show()

+----------+----------------+-----------+
| namespace|       tableName|isTemporary|
+----------+----------------+-----------+
|company_db|employee_details|      false|
|company_db| employee_master|      false|
|          |     departments|       true|
|          |       employees|       true|
|          |           sales|       true|
+----------+----------------+-----------+



In [ ]:
spark.sql(""" create or replace view high_salary_employees as select emp_id, name, department, salary from employee_details where salary > 70000  """)

DataFrame[]

In [ ]:
spark.sql("select * from high_salary_employees").show()

+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     4|Priya|   Finance| 80000|
|     7|Meera|   Finance| 82000|
|     3|Arjun|        IT| 75000|
+------+-----+----------+------+



In [ ]:
spark.sql(" select name, city from employees where department = 'IT' ").show()

In [ ]:
spark.sql("select * from employees where city in ('Hyderabad', 'Bangalore')").show()

In [ ]:
spark.sql(""" select e.emp_id, e.name, e.department, d.dept_full_name
          from employees e inner join departments d
          on e.department= d.department""").show()

In [ ]:
spark.sql(""" select e.emp_id, e.name, e.department, d.dept_full_name
          from employees e left join departments d
          on e.department= d.department""").show()

In [ ]:
spark.sql("""
          select count(*) as total_employees,
          sum(salary) as total_salary,
          avg(salary) as avg_salary,
          max(salary) as max_salary,
          min(salary) as min_salary
          from employees""").show()

In [ ]:
spark.sql(" select department, count(*) as total_emp from employees group by department").show()

In [ ]:
spark.sql(" select department, avg(salary) as avg_salary from employees group by department").show()

In [ ]:
spark.sql(" select e.name, sum(s.quantity * s.price) as total_sales from employees e join sales s on e.emp_id = s.emp_id group by e.name").show()

In [ ]:
spark.sql(" select department, avg(salary) as avg_salary from employees group by department having avg_salary > 65000").show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|   Finance|   81000.0|
+----------+----------+



In [ ]:
spark.sql("""select product, sum(quantity) as total_quantity
             from sales group by product having total_quantity > 2 """).show()

In [ ]:
spark.sql(" select * from employees order by salary desc").show()

In [ ]:
spark.sql(""" select product, sum(quantity * price) as revenue from sales group by product
              order by revenue desc   """).show()

In [ ]:
spark.sql(""" select
              emp_id,
              name,
              department,
              salary,
              rank() over (partition by department order by salary desc) as salary_rank
              from employees
              """).show()

+------+-----+----------+------+-----------+
|emp_id| name|department|salary|salary_rank|
+------+-----+----------+------+-----------+
|     7|Meera|   Finance| 82000|          1|
|     4|Priya|   Finance| 80000|          2|
|     2|Sneha|        HR| 60000|          1|
|     6| Amit|        HR| 58000|          2|
|     3|Arjun|        IT| 75000|          1|
|     1|Rahul|        IT| 70000|          2|
|     5|Karan|        IT| 50000|          3|
+------+-----+----------+------+-----------+



In [ ]:
spark.sql("""select
            emp_id,
            name,
            department,
            salary,
            row_number() over (partition by department order by salary) as row_num
            from employees """).show()

+------+-----+----------+------+-------+
|emp_id| name|department|salary|row_num|
+------+-----+----------+------+-------+
|     4|Priya|   Finance| 80000|      1|
|     7|Meera|   Finance| 82000|      2|
|     6| Amit|        HR| 58000|      1|
|     2|Sneha|        HR| 60000|      2|
|     5|Karan|        IT| 50000|      1|
|     1|Rahul|        IT| 70000|      2|
|     3|Arjun|        IT| 75000|      3|
+------+-----+----------+------+-------+



In [ ]:
spark.sql("""select
             emp_id,
             name,
             department,
             salary,
             dense_rank() over (partition by department order by salary desc) as dense_rank
             from employees""").show()

+------+-----+----------+------+----------+
|emp_id| name|department|salary|dense_rank|
+------+-----+----------+------+----------+
|     7|Meera|   Finance| 82000|         1|
|     4|Priya|   Finance| 80000|         2|
|     2|Sneha|        HR| 60000|         1|
|     6| Amit|        HR| 58000|         2|
|     3|Arjun|        IT| 75000|         1|
|     1|Rahul|        IT| 70000|         2|
|     5|Karan|        IT| 50000|         3|
+------+-----+----------+------+----------+

